# 04 · Priority ranking, tested on unseen years (Iowa corn and soybeans)

**Direction:** Soil and Crop Resilience Planning. **User:** Soil Conservation Office.
**Decision:** which counties to prioritise for soil programs.

Notebook 03 found no clear association between soil water storage and drought-year losses (every range included zero). So the ranking here is built on **observed drought exposure and drought-year losses**, with storage shown as a comparison score only. Everything is built on **TRAIN = 2000–2015** and tested on **TEST = 2016–2025**, years the scores never saw.

**Strict trend (no leakage).** The master's `yield_anomaly_pct` uses a trend fitted on all 26 years, so it is not used here. Instead, for each county × crop, a linear trend of `yield_per_acre` on year is fitted on TRAIN years only, then extrapolated to TEST years. Anomaly = 100 × (yield − fitted) / fitted. **Relative loss** = county anomaly − statewide median anomaly that year (same crop), which removes the common weather shock so counties are ranked against each other.

**Priority scores (TRAIN only, per county).** Higher priority = more deserving of a program.
1. `drought_freq` — share of TRAIN years at D2+. Higher → higher priority.
2. `drought_loss` — mean TRAIN anomaly over the county's D2+ years (needs ≥ 3 such years). More negative → higher priority.
3. `storage` — `aws_100cm_mm`. Lower → higher priority. Comparison only.
4. `combined` — average of the within-Iowa percent ranks of (1) and (2), oriented so higher = higher priority.

**Test.** For each TEST year with ≥ 15 D2+ counties: Spearman correlation between the training priority and that year's relative loss (sign set so positive = the score correctly predicts worse losses), and the share of top-third-priority counties that landed in the worst third of relative loss (chance ≈ 33%). Averaged over TEST years with a 95% range from resampling TEST years.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import spearmanr
from src.data_loader import load_master, load_counties

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

df = load_master()
print(f"{len(df):,} rows × {len(df.columns)} columns")

17,056 rows × 21 columns


In [2]:
# ---- Column names (master dataset) ----
COL_FIPS = "fips"
COL_COUNTY = "county_name"
COL_YEAR = "year"
COL_STATE = "state_alpha"
COL_CROP = "crop"
COL_YIELD = "yield_per_acre"
COL_SEV = "max_drought_severity"        # 0-5; 3 = D2, 4 = D3, 5 = D4
COL_WEEKS_D2 = "weeks_in_d2_plus"
COL_HEAT = "extreme_heat_days"
COL_AWS = "aws_100cm_mm"
COL_LON = "lon"                         # county centroid longitude, from counties.geojson

# ---- Derived columns created in this notebook ----
COL_FITTED = "fitted_strict"            # trend value from the TRAIN-only fit
COL_ANOM = "anomaly_strict"             # 100 * (yield - fitted) / fitted
COL_REL = "relative_loss"               # anomaly_strict - statewide median anomaly_strict that year (per crop)

# ---- Split ----
TRAIN_YEARS = range(2000, 2016)          # 2000-2015 inclusive
TEST_YEARS = range(2016, 2026)           # 2016-2025 inclusive
MIN_TRAIN_YEARS_FOR_TREND = 8

# ---- Drought ----
SEV_D2 = 3
DROUGHT_SEV = SEV_D2                     # D2+
MIN_D2_TRAIN_YEARS_FOR_LOSS = 3          # for the drought_loss score
MIN_COUNTIES_PER_TEST_YEAR = 15
MIN_TEST_YEARS_WARN = 4

# ---- Scope ----
FOCUS_STATE = "IA"
FOCUS_CROPS = ["CORN", "SOYBEANS"]

# ---- Bootstrap ----
N_BOOT = 5000
BOOT_SEED = 20261017
CI = (2.5, 97.5)
TOP_SHARE = 1 / 3                        # "top third" of priority, "worst third" of loss

# ---- Priority scores: name -> (column holding the raw value, direction)
# direction +1: higher raw value = higher priority; -1: lower raw value = higher priority
SCORES = {
    "drought_freq": ("drought_freq", +1),   # more D2+ years -> higher priority
    "drought_loss": ("drought_loss", -1),   # more negative drought-year anomaly -> higher priority
    "storage":      ("storage", -1),        # less soil water storage -> higher priority (comparison only)
    "combined":     ("combined", +1),       # already oriented: higher = higher priority
}

## Data

In [3]:
ia = df[(df[COL_STATE] == FOCUS_STATE) & (df[COL_CROP].isin(FOCUS_CROPS)) & df[COL_YIELD].notna()].copy()
assert ia[COL_FIPS].map(type).eq(str).all() and ia[COL_FIPS].str.len().eq(5).all(), "fips must be 5-char text"
assert ia[COL_SEV].notna().all()
ia["split"] = np.where(ia[COL_YEAR].isin(TRAIN_YEARS), "train", np.where(ia[COL_YEAR].isin(TEST_YEARS), "test", "other"))
assert (ia["split"] != "other").all()
print(ia.groupby([COL_CROP, "split"]).agg(rows=(COL_FIPS, "size"), counties=(COL_FIPS, "nunique"), years=(COL_YEAR, "nunique")))

geo = load_counties()
def centroid_lon(feature):
    g = feature["geometry"]
    polys = g["coordinates"] if g["type"] == "Polygon" else [p for mp in g["coordinates"] for p in mp]
    pts = np.array([pt for ring in polys for pt in ring])
    return pts[:, 0].mean()
lon = pd.DataFrame([(str(f["id"]), centroid_lon(f)) for f in geo["features"] if str(f["id"]).startswith("19")],
                   columns=[COL_FIPS, COL_LON])
ia = ia.merge(lon, on=COL_FIPS, how="left")
assert ia[COL_LON].notna().all()

                rows  counties  years
crop     split                       
CORN     test    890        99     10
         train  1580        99     16
SOYBEANS test    855        99     10
         train  1582        99     16


## Strict trend: fitted on TRAIN years only, extrapolated to TEST years

In [4]:
def fit_train_trend(g):
    """Linear trend yield ~ year on TRAIN rows of one county x crop. Returns (slope, intercept, n_train)."""
    t = g[g["split"] == "train"]
    if len(t) < MIN_TRAIN_YEARS_FOR_TREND:
        return pd.Series({"slope": np.nan, "intercept": np.nan, "n_train": len(t)})
    slope, intercept = np.polyfit(t[COL_YEAR].to_numpy(float), t[COL_YIELD].to_numpy(float), 1)
    return pd.Series({"slope": slope, "intercept": intercept, "n_train": len(t)})

trend = ia.groupby([COL_CROP, COL_FIPS]).apply(fit_train_trend, include_groups=False).reset_index()
dropped_trend = trend[trend["slope"].isna()]
print(f"County x crop pairs: {len(trend)}; dropped for < {MIN_TRAIN_YEARS_FOR_TREND} training years: {len(dropped_trend)}")
if len(dropped_trend):
    display(dropped_trend)
print("Training years per county (min / median / max), by crop:")
display(trend.groupby(COL_CROP)["n_train"].agg(["min", "median", "max", "count"]))

ia = ia.merge(trend, on=[COL_CROP, COL_FIPS], how="left")
ia = ia[ia["slope"].notna()].copy()
ia[COL_FITTED] = ia["intercept"] + ia["slope"] * ia[COL_YEAR]
ia[COL_ANOM] = 100 * (ia[COL_YIELD] - ia[COL_FITTED]) / ia[COL_FITTED]

# Relative loss: remove the statewide shock of the year (median over all reporting counties, same crop)
year_median = ia.groupby([COL_CROP, COL_YEAR])[COL_ANOM].median().rename("state_median_anom")
ia = ia.merge(year_median, on=[COL_CROP, COL_YEAR], how="left")
ia[COL_REL] = ia[COL_ANOM] - ia["state_median_anom"]
assert ia[[COL_FITTED, COL_ANOM, COL_REL]].notna().all().all()

County x crop pairs: 198; dropped for < 8 training years: 0
Training years per county (min / median / max), by crop:


,min,median,max,count
crop,,,,
CORN,15.0,16.0,16.0,99
SOYBEANS,15.0,16.0,16.0,99


In [5]:
# Sanity check on the extrapolation: statewide median strict anomaly by year.
# TRAIN years are in-sample (should hover near 0). TEST years are extrapolated 1-10 years beyond the fit.
chk = (ia.groupby([COL_CROP, COL_YEAR])
         .agg(counties=(COL_FIPS, "nunique"),
              median_anomaly=(COL_ANOM, "median"),
              share_D2plus=(COL_SEV, lambda s: (s >= DROUGHT_SEV).mean()))
         .round(2))
display(chk.unstack(COL_CROP).swaplevel(axis=1).sort_index(axis=1).loc[list(TEST_YEARS)])
print("Mean of the yearly median strict anomaly in TEST years, by crop:")
print(chk.loc[(slice(None), list(TEST_YEARS)), "median_anomaly"].groupby(COL_CROP).mean().round(2))
print("\nIf these are far from zero, the TRAIN-only trend extrapolates poorly. relative_loss removes this statewide")
print("shift, so the ranking test is not affected by it, but the raw strict anomaly in TEST years is.")

crop     CORN                             SOYBEANS                            
     counties median_anomaly share_D2plus counties median_anomaly share_D2plus
year                                                                          
2016       99          15.03         0.00       99          13.74         0.00
2017       95          13.65         0.27       96           6.00         0.31
2018       93          11.10         0.15       96           4.31         0.18
2019       88           9.56         0.00       82           1.02         0.00
2020       95          -0.34         0.52       86          -0.82         0.51
2021       84          10.91         0.68       76          14.31         0.70
2022       97          10.74         0.45       97           5.44         0.63
2023       87           8.76         0.99       82           2.97         0.99
2024       73          16.61         0.33       68           4.84         0.72
2025       79          13.16         0.00       73           9.96         0.00

Mean of the yearly median strict anomaly in TEST years, by crop:
crop
CORN        10.92
SOYBEANS     6.18
Name: median_anomaly, dtype: float64

If these are far from zero, the TRAIN-only trend extrapolates poorly. relative_loss removes this statewide
shift, so the ranking test is not affected by it, but the raw strict anomaly in TEST years is.


## Priority scores from TRAIN years only

In [6]:
train = ia[ia["split"] == "train"]

def county_scores(g):
    d2 = g[g[COL_SEV] >= DROUGHT_SEV]
    return pd.Series({
        "n_train_years": g[COL_YEAR].nunique(),
        "n_d2_train_years": d2[COL_YEAR].nunique(),
        "drought_freq": (g[COL_SEV] >= DROUGHT_SEV).mean(),                      # share of TRAIN years at D2+
        "drought_loss": d2[COL_ANOM].mean() if len(d2) >= MIN_D2_TRAIN_YEARS_FOR_LOSS else np.nan,  # mean anomaly in D2+ years
        "storage": g[COL_AWS].iloc[0],
    })

scores = train.groupby([COL_CROP, COL_FIPS]).apply(county_scores, include_groups=False).reset_index()
scores = scores.merge(ia[[COL_FIPS, COL_COUNTY]].drop_duplicates(), on=COL_FIPS, how="left")

# Combined: average of within-Iowa percent ranks, oriented so higher = higher priority.
#   drought_freq: higher share -> higher priority -> rank ascending
#   drought_loss: more negative -> higher priority -> rank of (-drought_loss)
scores["pct_rank_freq"] = scores.groupby(COL_CROP)["drought_freq"].rank(pct=True)
scores["pct_rank_loss"] = scores.groupby(COL_CROP)["drought_loss"].transform(lambda s: (-s).rank(pct=True))
scores["combined"] = scores[["pct_rank_freq", "pct_rank_loss"]].mean(axis=1, skipna=False)

for crop in FOCUS_CROPS:
    s = scores[scores[COL_CROP] == crop]
    dropped_loss = s[s["drought_loss"].isna()]
    print(f"{crop}: {len(s)} counties scored; drought_loss missing for {len(dropped_loss)} counties "
          f"(< {MIN_D2_TRAIN_YEARS_FOR_LOSS} D2+ training years); combined therefore missing for the same {len(dropped_loss)}")
    if len(dropped_loss):
        print("   dropped:", ", ".join(f"{n.replace(' County','')} ({int(k)} D2+ yrs)"
                                       for n, k in zip(dropped_loss[COL_COUNTY], dropped_loss["n_d2_train_years"])))
print()
print("Score distributions (TRAIN):")
display(scores.groupby(COL_CROP)[["drought_freq", "drought_loss", "storage", "combined"]].describe().round(3).T)

CORN: 99 counties scored; drought_loss missing for 5 counties (< 3 D2+ training years); combined therefore missing for the same 5
   dropped: Buchanan (2 D2+ yrs), Clayton (2 D2+ yrs), Fayette (2 D2+ yrs), Howard (2 D2+ yrs), Winneshiek (2 D2+ yrs)
SOYBEANS: 99 counties scored; drought_loss missing for 5 counties (< 3 D2+ training years); combined therefore missing for the same 5
   dropped: Buchanan (2 D2+ yrs), Clayton (2 D2+ yrs), Fayette (2 D2+ yrs), Howard (2 D2+ yrs), Winneshiek (2 D2+ yrs)

Score distributions (TRAIN):


crop                   CORN  SOYBEANS
drought_freq count   99.000    99.000
             mean     0.293     0.317
             std      0.082     0.092
             min      0.125     0.125
             25%      0.250     0.250
             50%      0.312     0.312
             75%      0.375     0.375
             max      0.467     0.467
drought_loss count   94.000    94.000
             mean    -8.738    -7.929
             std      3.925     4.642
             min    -18.778   -20.913
             25%    -11.278   -11.054
             50%     -8.181    -7.311
             75%     -5.978    -5.079
             max     -0.305     0.352
storage      count   99.000    99.000
             mean   179.242   179.242
             std     11.419    11.419
             min    154.245   154.245
             25%    172.001   172.001
             50%    178.999   178.999
             75%    184.778   184.778
             max    205.757   205.757
combined     count   94.000    94.000
             mean     0.518     0.518
             std      0.155     0.148
             min      0.105     0.100
             25%      0.410     0.431
             50%      0.534     0.536
             75%      0.621     0.601
             max      0.865     0.869

In [7]:
# Priority = raw score times its direction, so HIGHER priority is always HIGHER number in the test below.
for name, (col, direction) in SCORES.items():
    scores[f"priority_{name}"] = direction * scores[col]

print("Spearman correlation between the four priority orderings (TRAIN), by crop:")
for crop in FOCUS_CROPS:
    s = scores[scores[COL_CROP] == crop]
    display(s[[f"priority_{n}" for n in SCORES]].corr(method="spearman").round(2).rename_axis(crop))

Spearman correlation between the four priority orderings (TRAIN), by crop:


,priority_drought_freq,priority_drought_loss,priority_storage,priority_combined
CORN,,,,
priority_drought_freq,1.00,-0.38,-0.21,0.49
priority_drought_loss,-0.38,1.00,0.07,0.59
priority_storage,-0.21,0.07,1.00,-0.09
priority_combined,0.49,0.59,-0.09,1.00


,priority_drought_freq,priority_drought_loss,priority_storage,priority_combined
SOYBEANS,,,,
priority_drought_freq,1.00,-0.44,-0.21,0.44
priority_drought_loss,-0.44,1.00,0.14,0.59
priority_storage,-0.21,0.14,1.00,0.03
priority_combined,0.44,0.59,0.03,1.00


In [8]:
# Top-10 priority counties per crop by the combined score, for reference
for crop in FOCUS_CROPS:
    s = scores[scores[COL_CROP] == crop].sort_values("priority_combined", ascending=False)
    print(f"{crop}: top 10 by combined priority (TRAIN years only)")
    display(s[[COL_FIPS, COL_COUNTY, "n_d2_train_years", "drought_freq", "drought_loss", "storage", "combined"]]
              .head(10).round(3).reset_index(drop=True))

CORN: top 10 by combined priority (TRAIN years only)


,fips,county_name,n_d2_train_years,drought_freq,drought_loss,storage,combined
0,19051,Davis County,7.0,0.438,-11.370,163.360,0.865
1,19161,Sac County,6.0,0.375,-13.789,190.467,0.856
2,19135,Monroe County,7.0,0.467,-9.201,166.778,0.824
3,19047,Crawford County,6.0,0.375,-11.311,204.654,0.792
4,19027,Carroll County,6.0,0.375,-11.177,190.023,0.786
5,19149,Plymouth County,5.0,0.312,-15.086,205.757,0.748
6,19175,Union County,6.0,0.400,-8.489,178.999,0.739
7,19133,Monona County,6.0,0.375,-8.935,184.686,0.723
8,19177,Van Buren County,6.0,0.375,-8.882,165.798,0.717
9,19111,Lee County,5.0,0.312,-12.840,170.519,0.706


SOYBEANS: top 10 by combined priority (TRAIN years only)


,fips,county_name,n_d2_train_years,drought_freq,drought_loss,storage,combined
0,19161,Sac County,7.0,0.438,-12.296,190.467,0.869
1,19135,Monroe County,7.0,0.438,-10.895,166.778,0.827
2,19051,Davis County,7.0,0.438,-9.782,163.360,0.805
3,19027,Carroll County,7.0,0.438,-9.293,190.023,0.789
4,19073,Greene County,7.0,0.438,-8.570,183.228,0.768
5,19007,Appanoose County,6.0,0.375,-12.585,162.188,0.764
6,19175,Union County,7.0,0.438,-8.427,178.999,0.757
7,19029,Cass County,6.0,0.375,-11.416,189.485,0.748
8,19025,Calhoun County,7.0,0.438,-7.962,181.301,0.747
9,19001,Adair County,6.0,0.375,-11.277,181.112,0.732


## Test on TEST years (2016–2025)

In [9]:
test = ia[ia["split"] == "test"]

def usable_test_years(crop):
    n = (test[(test[COL_CROP] == crop) & (test[COL_SEV] >= DROUGHT_SEV)]
            .groupby(COL_YEAR)[COL_FIPS].nunique())
    return n[n >= MIN_COUNTIES_PER_TEST_YEAR], n[n < MIN_COUNTIES_PER_TEST_YEAR]

for crop in FOCUS_CROPS:
    kept, dropped = usable_test_years(crop)
    print(f"{crop}: TEST years with >= {MIN_COUNTIES_PER_TEST_YEAR} D2+ counties: {len(kept)} -> "
          + ", ".join(f"{y} (n={n})" for y, n in kept.items()))
    print(f"   TEST years with fewer: " + (", ".join(f"{y} (n={n})" for y, n in dropped.items()) or "none")
          + "; TEST years with zero D2+ counties: "
          + ", ".join(str(y) for y in TEST_YEARS if y not in kept.index and y not in dropped.index))
    if len(kept) < MIN_TEST_YEARS_WARN:
        print(f"   *** WARNING: only {len(kept)} usable TEST years (fewer than {MIN_TEST_YEARS_WARN}). "
              "Any average over them is too thin to mean much. ***")
    print()

CORN: TEST years with >= 15 D2+ counties: 6 -> 2017 (n=26), 2020 (n=49), 2021 (n=57), 2022 (n=44), 2023 (n=86), 2024 (n=24)
   TEST years with fewer: 2018 (n=14); TEST years with zero D2+ counties: 2016, 2019, 2025

SOYBEANS: TEST years with >= 15 D2+ counties: 7 -> 2017 (n=30), 2018 (n=17), 2020 (n=44), 2021 (n=53), 2022 (n=61), 2023 (n=81), 2024 (n=49)
   TEST years with fewer: none; TEST years with zero D2+ counties: 2016, 2019, 2025



In [10]:
def year_bootstrap(values, n_boot=N_BOOT, seed=BOOT_SEED, ci=CI):
    v = np.asarray(pd.Series(values).dropna(), dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(v), size=(n_boot, len(v)))
    means = v[idx].mean(axis=1)
    lo, hi = np.percentile(means, ci)
    return v.mean(), lo, hi, len(v)


def evaluate_year(d, priority_col):
    """d = one TEST year's D2+ counties (one crop) joined to the priority score.
    Returns Spearman (positive = score predicts worse losses) and the top-third hit share."""
    d = d[[priority_col, COL_REL]].dropna()
    n = len(d)
    if n < MIN_COUNTIES_PER_TEST_YEAR:
        return pd.Series({"n": n, "spearman": np.nan, "top_third_hit": np.nan})
    # Higher priority should go with MORE NEGATIVE relative loss, so correlate priority with -relative_loss.
    rho = spearmanr(d[priority_col], -d[COL_REL]).correlation
    k = max(1, int(round(TOP_SHARE * n)))
    top_priority = set(d.nlargest(k, priority_col).index)           # highest priority
    worst_loss = set(d.nsmallest(k, COL_REL).index)                  # most negative relative loss
    hit = len(top_priority & worst_loss) / k
    return pd.Series({"n": n, "spearman": rho, "top_third_hit": hit})


def run_test(crop):
    kept, _ = usable_test_years(crop)
    t = test[(test[COL_CROP] == crop) & (test[COL_SEV] >= DROUGHT_SEV) & test[COL_YEAR].isin(kept.index)]
    s = scores[scores[COL_CROP] == crop].set_index(COL_FIPS)
    per_year, summary = {}, []
    for name in SCORES:
        pcol = f"priority_{name}"
        tt = t.join(s[[pcol]], on=COL_FIPS)
        py = tt.groupby(COL_YEAR).apply(lambda d: evaluate_year(d, pcol), include_groups=False)
        per_year[name] = py
        for metric, chance in [("spearman", 0.0), ("top_third_hit", TOP_SHARE)]:
            m, lo, hi, n = year_bootstrap(py[metric])
            summary.append({"crop": crop, "score": name, "metric": metric, "mean": m, "ci_low": lo, "ci_high": hi,
                            "n_years": n, "chance": chance, "beats_chance (range above chance)": lo > chance,
                            "years_above_chance": int((py[metric] > chance).sum())})
    return per_year, pd.DataFrame(summary)


results = {crop: run_test(crop) for crop in FOCUS_CROPS}

In [11]:
def fmt(r):
    return f"{r['mean']:+.2f} [{r['ci_low']:+.2f}, {r['ci_high']:+.2f}]"

for crop in FOCUS_CROPS:
    per_year, summary = results[crop]
    print(f"===== {crop} =====")
    print("Per-year Spearman (positive = priority score predicts worse relative loss), over that year's D2+ counties:")
    sp = pd.concat({n: py["spearman"] for n, py in per_year.items()}, axis=1)
    sp.insert(0, "n_counties", per_year["drought_freq"]["n"].astype(int))
    sp["n_with_loss_score"] = per_year["drought_loss"]["n"].astype(int)
    display(sp.round(2))
    print("Per-year share of top-third-priority counties that were in the worst third by relative loss (chance = 0.33):")
    hit = pd.concat({n: py["top_third_hit"] for n, py in per_year.items()}, axis=1)
    display(hit.round(2))
    print("Summary over TEST years (mean [95% range from resampling TEST years]):")
    v = summary.copy()
    v["mean [95% range]"] = v.apply(fmt, axis=1)
    display(v.pivot(index="score", columns="metric", values="mean [95% range]")
             .join(v[v["metric"] == "spearman"].set_index("score")[["n_years"]])
             .join(v.pivot(index="score", columns="metric", values="years_above_chance").add_prefix("yrs>chance: "))
             .join(v.pivot(index="score", columns="metric", values="beats_chance (range above chance)").add_prefix("range>chance: "))
             .loc[list(SCORES)])
    print()

===== CORN =====
Per-year Spearman (positive = priority score predicts worse relative loss), over that year's D2+ counties:


,n_counties,drought_freq,drought_loss,storage,combined,n_with_loss_score
year,,,,,,
2017,26,-0.05,0.26,-0.38,0.21,25
2020,49,-0.02,0.04,-0.02,-0.05,49
2021,57,-0.14,-0.38,-0.16,-0.47,52
2022,44,-0.23,-0.01,-0.77,-0.08,44
2023,86,-0.34,0.12,-0.37,-0.12,81
2024,24,-0.68,0.63,0.11,0.09,19


Per-year share of top-third-priority counties that were in the worst third by relative loss (chance = 0.33):


,drought_freq,drought_loss,storage,combined
year,,,,
2017,0.33,0.50,0.22,0.38
2020,0.44,0.31,0.25,0.31
2021,0.26,0.24,0.16,0.18
2022,0.13,0.47,0.00,0.40
2023,0.34,0.41,0.24,0.33
2024,0.00,0.50,0.38,0.17


Summary over TEST years (mean [95% range from resampling TEST years]):


...

,n_counties,drought_freq,drought_loss,storage,combined,n_with_loss_score
year,,,,,,
2017,30,0.20,-0.15,0.15,-0.00,29
2018,17,0.08,-0.01,-0.14,0.01,17
2020,44,0.12,0.40,-0.28,0.52,44
2021,53,-0.39,-0.12,-0.08,-0.51,49
2022,61,0.09,-0.09,-0.30,-0.03,61
2023,81,-0.57,0.24,-0.05,-0.23,76
2024,49,-0.48,-0.13,-0.00,-0.51,45


Per-year share of top-third-priority counties that were in the worst third by relative loss (chance = 0.33):


,drought_freq,drought_loss,storage,combined
year,,,,
2017,0.60,0.30,0.50,0.40
2018,0.17,0.33,0.17,0.33
2020,0.53,0.53,0.20,0.67
2021,0.17,0.31,0.22,0.12
2022,0.35,0.25,0.30,0.30
2023,0.11,0.60,0.48,0.28
2024,0.12,0.33,0.31,0.13


Summary over TEST years (mean [95% range from resampling TEST years]):


,spearman,top_third_hit,n_years,yrs>chance: spearman,yrs>chance: top_third_hit,range>chance: spearman,range>chance: top_third_hit
score,,,,,,,
drought_freq,"-0.13 [-0.37, +0.07]","+0.29 [+0.16, +0.44]",7,4,3,False,False
drought_loss,"+0.02 [-0.11, +0.17]","+0.38 [+0.30, +0.47]",7,2,2,False,False
storage,"-0.10 [-0.21, +0.01]","+0.31 [+0.23, +0.40]",7,1,2,False,False
combined,"-0.11 [-0.34, +0.15]","+0.32 [+0.20, +0.45]",7,2,2,False,False


In [12]:
# All four scores, both crops, side by side
allsum = pd.concat([results[c][1] for c in FOCUS_CROPS], ignore_index=True)
allsum["mean [95% range]"] = allsum.apply(fmt, axis=1)
side = allsum.pivot(index=["metric", "score"], columns="crop", values="mean [95% range]").loc[
    [(m, s) for m in ["spearman", "top_third_hit"] for s in SCORES]]
display(side)
print("Chance level: spearman 0.00, top_third_hit 0.33. 'Beats chance' = the whole 95% range sits above chance.")

fig = go.Figure()
for i, crop in enumerate(FOCUS_CROPS):
    s = allsum[(allsum["crop"] == crop) & (allsum["metric"] == "spearman")]
    fig.add_trace(go.Scatter(x=s["score"], y=s["mean"], mode="markers", name=crop,
                             error_y=dict(type="data", symmetric=False,
                                          array=s["ci_high"] - s["mean"], arrayminus=s["mean"] - s["ci_low"]),
                             marker=dict(size=10)))
fig.add_hline(y=0, line_dash="dot", line_color="grey", annotation_text="chance")
fig.update_layout(title="Mean Spearman between TRAIN priority and TEST-year relative loss (D2+ counties), 95% year-bootstrap range",
                  yaxis_title="Spearman (positive = predicts losses)", height=400)
fig.show()

crop                                        CORN              SOYBEANS
metric        score                                                   
spearman      drought_freq  -0.24 [-0.44, -0.09]  -0.13 [-0.37, +0.07]
              drought_loss  +0.11 [-0.13, +0.36]  +0.02 [-0.11, +0.17]
              storage       -0.27 [-0.51, -0.04]  -0.10 [-0.21, +0.01]
              combined      -0.07 [-0.25, +0.08]  -0.11 [-0.34, +0.15]
top_third_hit drought_freq  +0.25 [+0.12, +0.36]  +0.29 [+0.16, +0.44]
              drought_loss  +0.40 [+0.32, +0.48]  +0.38 [+0.30, +0.47]
              storage       +0.21 [+0.11, +0.29]  +0.31 [+0.23, +0.40]
              combined      +0.29 [+0.22, +0.36]  +0.32 [+0.20, +0.45]

Chance level: spearman 0.00, top_third_hit 0.33. 'Beats chance' = the whole 95% range sits above chance.


## Secondary check: notebook 03 spec C storage estimate, TRAIN years vs TEST years

Same specification as notebook 03 spec C (anomaly ~ storage + drought weeks + heat days + longitude, per D2+ year, ≥ 15 counties), but using the strict anomaly from this notebook. Slope in percentage points per +10 mm of storage. Note that TRAIN-year anomalies are in-sample for the trend; TEST-year anomalies are extrapolated.

In [13]:
AWS_UNIT = 10.0
SPEC_C = [COL_WEEKS_D2, COL_HEAT, COL_LON]

def ols_slope(d, controls):
    X = np.column_stack([np.ones(len(d)), d[[COL_AWS] + controls].to_numpy(float)])
    y = d[COL_ANOM].to_numpy(float)
    if np.linalg.matrix_rank(X) < X.shape[1]:
        return np.nan
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return beta[1] * AWS_UNIT

rows, per_year_c = [], {}
for crop in FOCUS_CROPS:
    for split in ["train", "test"]:
        sub = ia[(ia[COL_CROP] == crop) & (ia["split"] == split) & (ia[COL_SEV] >= DROUGHT_SEV)]
        n = sub.groupby(COL_YEAR)[COL_FIPS].nunique()
        kept = n[n >= MIN_COUNTIES_PER_TEST_YEAR]
        sub = sub[sub[COL_YEAR].isin(kept.index)]
        sl = sub.groupby(COL_YEAR).apply(lambda d: ols_slope(d, SPEC_C), include_groups=False)
        per_year_c[(crop, split)] = pd.DataFrame({"n_counties": kept, "slope_per_10mm": sl})
        m, lo, hi, k = year_bootstrap(sl)
        rows.append({"crop": crop, "years": split.upper(), "estimate_per_10mm": m, "ci_low": lo, "ci_high": hi,
                     "n_years": k, "years_used": ", ".join(map(str, kept.index)),
                     "years_positive": int((sl > 0).sum()), "years_negative": int((sl < 0).sum())})
secondary = pd.DataFrame(rows)
secondary["estimate [95% range]"] = secondary.apply(
    lambda r: f"{r['estimate_per_10mm']:+.2f} [{r['ci_low']:+.2f}, {r['ci_high']:+.2f}]", axis=1)
display(secondary.set_index(["crop", "years"])[["estimate [95% range]", "n_years", "years_positive", "years_negative", "years_used"]])

for key, t in per_year_c.items():
    print(key)
    display(t.round(2).T)

estimate [95% range]  n_years  years_positive  years_negative                                years_used
crop     years                                                                                                         
CORN     TRAIN  -0.01 [-2.04, +1.97]        7               4               3  2000, 2003, 2005, 2006, 2011, 2012, 2013
         TEST   -5.49 [-8.20, -2.60]        6               1               5        2017, 2020, 2021, 2022, 2023, 2024
SOYBEANS TRAIN  -0.25 [-2.19, +1.56]        7               3               4  2000, 2003, 2005, 2006, 2011, 2012, 2013
         TEST   -1.84 [-4.26, +1.14]        7               1               6  2017, 2018, 2020, 2021, 2022, 2023, 2024

('CORN', 'train')


year,2000,2003,2005,2006,2011,2012,2013
n_counties,83.00,92.00,26.00,51.00,16.00,99.00,64.00
slope_per_10mm,-3.79,1.19,-1.27,-3.33,0.86,4.41,1.87


('CORN', 'test')


year,2017,2020,2021,2022,2023,2024
n_counties,26.00,49.0,57.00,44.00,86.00,24.00
slope_per_10mm,-9.42,-5.0,-1.98,-7.63,-9.06,0.14


('SOYBEANS', 'train')


year,2000,2003,2005,2006,2011,2012,2013
n_counties,83.00,92.00,47.00,51.00,33.00,99.00,65.00
slope_per_10mm,-2.82,1.96,-4.44,-0.71,-0.28,1.59,2.98


('SOYBEANS', 'test')


year,2017,2018,2020,2021,2022,2023,2024
n_counties,30.00,17.00,44.00,53.00,61.00,81.00,49.00
slope_per_10mm,-5.75,6.13,-5.21,-0.98,-0.68,-4.29,-2.12


## Diagnostic: is the TRAIN-only trend extrapolation biased by region or storage?

The sanity check above showed the strict anomaly is about +11 pp (corn) and +6 pp (soybeans) too high on average in TEST years: yields rose faster after 2015 than the 2000–2015 line predicts. That alone would not matter, because relative loss removes the statewide shift. It matters if the extrapolation error differs systematically between counties. Three checks:

1. correlation of each county's TRAIN trend slope with storage and longitude;
2. mean TEST-year relative loss by storage tertile (should be near 0 in every tertile if there is no county-specific bias);
3. a **placebo**: the ranking test and the storage slope in TEST years with **zero** D2+ counties (2016, 2019, 2025). A score that "predicts" relative loss in years with no drought is not detecting drought vulnerability.

In [14]:
cty = (ia.groupby([COL_CROP, COL_FIPS])
         .agg(trend_slope=("slope", "first"), storage=(COL_AWS, "first"), lon=(COL_LON, "first"))
         .reset_index())
cty["storage_tertile"] = cty.groupby(COL_CROP)["storage"].transform(
    lambda s: pd.qcut(s, 3, labels=["low", "middle", "high"]).astype(str))

print("1. County TRAIN trend slope (bu/acre/year) vs storage and longitude:")
display(cty.groupby(COL_CROP).apply(lambda g: pd.Series({
    "corr(trend slope, storage)": g["trend_slope"].corr(g["storage"]),
    "corr(trend slope, longitude)": g["trend_slope"].corr(g["lon"]),
    "trend slope mean": g["trend_slope"].mean(),
    "trend slope sd": g["trend_slope"].std()}), include_groups=False).round(2))

print("2. Mean strict anomaly and relative loss in TEST years by storage tertile (all TEST years, all reporting counties):")
tt = test.merge(cty[[COL_CROP, COL_FIPS, "storage_tertile"]], on=[COL_CROP, COL_FIPS])
display(tt.groupby([COL_CROP, "storage_tertile"])
          .agg(county_years=(COL_FIPS, "size"), mean_anomaly=(COL_ANOM, "mean"), mean_relative_loss=(COL_REL, "mean"))
          .round(2).reindex(["low", "middle", "high"], level="storage_tertile"))

1. County TRAIN trend slope (bu/acre/year) vs storage and longitude:


,"corr(trend slope, storage)","corr(trend slope, longitude)",trend slope mean,trend slope sd
crop,,,,
CORN,0.70,-0.44,1.13,1.02
SOYBEANS,0.51,-0.05,0.53,0.21


2. Mean strict anomaly and relative loss in TEST years by storage tertile (all TEST years, all reporting counties):


county_years  mean_anomaly  mean_relative_loss
crop     storage_tertile                                                
CORN     low                       293         19.13                8.34
         middle                    298         11.50                0.72
         high                      299          5.87               -4.90
SOYBEANS low                       289          9.78                3.52
         middle                    282          6.58                0.55
         high                      284          4.55               -1.67

In [15]:
print("3. Placebo on TEST years with zero D2+ counties (all reporting counties that year):")
print("   Spearman sign as in the main test: positive = priority score goes with worse relative loss.")
PLACEBO_CONTROLS = [COL_HEAT, COL_LON]   # weeks_in_d2_plus is 0 for every county in these years, so it is dropped
rows = []
for crop in FOCUS_CROPS:
    s = scores[scores[COL_CROP] == crop].set_index(COL_FIPS)
    tc = test[test[COL_CROP] == crop]
    zero_years = [y for y in TEST_YEARS if (tc.loc[tc[COL_YEAR] == y, COL_SEV] >= DROUGHT_SEV).sum() == 0]
    for y in zero_years:
        d = tc[tc[COL_YEAR] == y].join(s[[f"priority_{n}" for n in SCORES]], on=COL_FIPS)
        r = {"crop": crop, "year": y, "n_counties": len(d)}
        for n in SCORES:
            dd = d[[f"priority_{n}", COL_REL]].dropna()
            r[f"spearman_{n}"] = spearmanr(dd[f"priority_{n}"], -dd[COL_REL]).correlation
        r["storage_slope_per_10mm (storage+heat+lon)"] = ols_slope(d, PLACEBO_CONTROLS)
        rows.append(r)
placebo = pd.DataFrame(rows).set_index(["crop", "year"]).round(2)
display(placebo)

3. Placebo on TEST years with zero D2+ counties (all reporting counties that year):
   Spearman sign as in the main test: positive = priority score goes with worse relative loss.


n_counties  spearman_drought_freq  spearman_drought_loss  spearman_storage  spearman_combined  storage_slope_per_10mm (storage+heat+lon)
crop     year                                                                                                                                          
CORN     2016          99                  -0.27                  -0.14             -0.50              -0.34                                      -5.56
         2019          88                  -0.08                  -0.46             -0.24              -0.46                                      -4.58
         2025          79                  -0.29                  -0.08             -0.55              -0.29                                      -9.89
SOYBEANS 2016          99                  -0.19                  -0.00             -0.18              -0.08                                      -5.18
         2019          82                  -0.42                  -0.05             -0.04              -0.42                                      -3.15
         2025          73                  -0.39                  -0.02             -0.38              -0.38                                      -6.29

### Diagnostic: regression to the mean, separated from the trend method

**This cell deliberately uses the master's `yield_anomaly_pct`** (trend fitted on all 26 years), which is leaky and is used nowhere else in this notebook and in no estimate. It is used here only to separate two explanations of the placebo failure: (a) the TRAIN-only trend extrapolates unevenly, and (b) counties with a bad early drought record did relatively better later regardless of weather (regression to the mean). If the scores still "predict" relative loss in zero-drought years when the outcome is the all-years anomaly, then (b) is present, not only (a).

Relative loss here is `yield_anomaly_pct` minus the statewide median `yield_anomaly_pct` that year (same crop).

In [16]:
COL_ANOM_MASTER = "yield_anomaly_pct"
COL_REL_MASTER = "relative_loss_master"
ia[COL_REL_MASTER] = ia[COL_ANOM_MASTER] - ia.groupby([COL_CROP, COL_YEAR])[COL_ANOM_MASTER].transform("median")
test_m = ia[ia["split"] == "test"]

rows = []
for crop in FOCUS_CROPS:
    s = scores[scores[COL_CROP] == crop].set_index(COL_FIPS)
    tc = test_m[test_m[COL_CROP] == crop]
    for y in TEST_YEARS:
        d = tc[tc[COL_YEAR] == y].join(s[[f"priority_{n}" for n in SCORES]], on=COL_FIPS)
        r = {"crop": crop, "year": y, "n_counties": len(d), "D2+ counties": int((d[COL_SEV] >= DROUGHT_SEV).sum())}
        for n in SCORES:
            a = d[[f"priority_{n}", COL_REL]].dropna()
            b = d[[f"priority_{n}", COL_REL_MASTER]].dropna()
            r[f"{n}: strict"] = spearmanr(a[f"priority_{n}"], -a[COL_REL]).correlation
            r[f"{n}: master"] = spearmanr(b[f"priority_{n}"], -b[COL_REL_MASTER]).correlation
        rows.append(r)
rtm = pd.DataFrame(rows).set_index(["crop", "year"]).round(2)
print("Spearman (positive = score predicts worse relative loss) over ALL reporting counties in each TEST year,")
print("with the strict (TRAIN-only trend) outcome vs the master (all-years trend) outcome. Zero-drought years: 2016, 2019, 2025.")
display(rtm)

print("\nStorage slope per +10 mm (storage + heat + longitude) over all counties in each TEST year, strict vs master outcome:")
def _ols_y(d, ycol, controls):
    X = np.column_stack([np.ones(len(d)), d[[COL_AWS] + controls].to_numpy(float)])
    b, *_ = np.linalg.lstsq(X, d[ycol].to_numpy(float), rcond=None)
    return b[1] * AWS_UNIT
sl = (test_m.groupby([COL_CROP, COL_YEAR])
            .apply(lambda d: pd.Series({"strict": _ols_y(d, COL_ANOM, [COL_HEAT, COL_LON]),
                                        "master": _ols_y(d, COL_ANOM_MASTER, [COL_HEAT, COL_LON])}), include_groups=False)
            .round(2).unstack(COL_CROP).swaplevel(axis=1).sort_index(axis=1))
display(sl)
print("Reading: if the master-outcome storage slope sits near zero while the strict one is strongly negative, the strict")
print("storage result is a trend-extrapolation artifact. If the score correlations stay negative in zero-drought years")
print("under BOTH outcomes, regression to the mean is present as well.")

Spearman (positive = score predicts worse relative loss) over ALL reporting counties in each TEST year,
with the strict (TRAIN-only trend) outcome vs the master (all-years trend) outcome. Zero-drought years: 2016, 2019, 2025.


n_counties  D2+ counties  drought_freq: strict  drought_freq: master  drought_loss: strict  drought_loss: master  storage: strict  storage: master  combined: strict  combined: master
crop     year                                                                                                                                                                                        
CORN     2016          99             0                 -0.27                 -0.37                 -0.14                 -0.08            -0.50            -0.20             -0.34             -0.40
         2017          95            26                  0.12                  0.14                 -0.08                  0.08            -0.37            -0.07              0.08              0.23
         2018          93            14                 -0.10                 -0.04                 -0.19                 -0.25            -0.45            -0.23             -0.22             -0.25
         2019          88             0                 -0.08                  0.08                 -0.46                 -0.39            -0.24             0.18             -0.46             -0.29
         2020          95            49                 -0.04                 -0.03                 -0.09                  0.07            -0.50            -0.16             -0.14              0.01
         2021          84            57                 -0.22                 -0.17                 -0.26                 -0.12            -0.29             0.21             -0.36             -0.23
         2022          97            44                  0.28                  0.61                 -0.17                  0.00            -0.70            -0.25              0.14              0.52
         2023          87            86                 -0.35                 -0.41                  0.11                  0.27            -0.37            -0.13             -0.14             -0.06
         2024          73            24                 -0.27                 -0.34                 -0.07                 -0.07            -0.39            -0.24             -0.25             -0.32
         2025          79             0                 -0.29                 -0.32                 -0.08                  0.08            -0.55            -0.26             -0.29             -0.21
SOYBEANS 2016          99             0                 -0.19                 -0.07                 -0.00                 -0.05            -0.18            -0.04             -0.08             -0.05
         2017          96            30                 -0.13                 -0.08                  0.09                  0.17             0.20             0.43              0.09              0.19
         2018          96            17                 -0.23                 -0.06                 -0.03                  0.00            -0.30            -0.11             -0.16             -0.02
         2019          82             0                 -0.42                 -0.24                 -0.05                 -0.14            -0.04             0.05             -0.42             -0.40
         2020          86            44                  0.12                  0.40                  0.30                  0.31            -0.38            -0.16              0.45              0.63
         2021          76            53                 -0.41                 -0.26                  0.00                  0.04            -0.27            -0.10             -0.31             -0.18
         2022          97            61                  0.37                  0.63                 -0.27                 -0.32            -0.41            -0.28              0.05              0.18
         2023          82            81                 -0.57                 -0.52                  0.25                  0.34            -0.06             0.08             -0.22             -0.10
         2024          68            49                


Storage slope per +10 mm (storage + heat + longitude) over all counties in each TEST year, strict vs master outcome:


crop   CORN        SOYBEANS       
     master strict   master strict
year                              
2016  -1.44  -5.56    -2.39  -5.18
2017   1.34  -1.59     1.30  -0.72
2018   2.01  -0.80     1.74  -0.24
2019  -0.54  -4.58    -0.52  -3.15
2020  -3.18  -8.46    -1.12  -4.38
2021  -0.87  -5.78    -2.25  -6.46
2022   0.79  -4.99     0.93  -1.94
2023  -2.68  -9.11    -0.47  -3.61
2024   0.23  -4.59     0.80  -2.75
2025  -2.01  -9.89    -2.11  -6.29

Reading: if the master-outcome storage slope sits near zero while the strict one is strongly negative, the strict
storage result is a trend-extrapolation artifact. If the score correlations stay negative in zero-drought years
under BOTH outcomes, regression to the mean is present as well.


---
## Reading the results (plain language)

**How thin the test is.** Corn has 6 usable TEST years (2017, 2020–2024) and soybeans 7 (adds 2018). 2016, 2019 and 2025 had no D2+ county at all. 2023 alone supplies about a third of the test county-years. No 2012-scale event occurs in the test period. Six or seven years is enough to run the check but not enough for narrow ranges, and the ranges below are wide.

**Nothing beats chance on unseen years, for either crop.** "Beats chance" means the whole 95% range sits above chance (Spearman above 0, or top-third hit rate above 0.33).

Corn (6 years):
- `drought_loss` is the only score that leans the right way: Spearman +0.11 [−0.13, +0.36], hit rate 0.40 [0.32, 0.48]. The hit-rate range touches chance at its lower end. Four of six years are above chance, two below.
- `drought_freq` points the **wrong** way: Spearman −0.24 [−0.44, −0.09], hit rate 0.25 [0.12, 0.36]. Counties that were in drought most often in 2000–2015 did relatively *better* in 2016–2025 drought years.
- `storage` alone also points the wrong way: Spearman −0.27 [−0.51, −0.04], hit rate 0.21 [0.11, 0.29].
- `combined`: Spearman −0.07 [−0.25, +0.08], hit rate 0.29 [0.22, 0.36]. Averaging a weak right-way score with a wrong-way score gives noise.

Soybeans (7 years):
- `drought_loss`: Spearman +0.02 [−0.11, +0.17], hit rate 0.38 [0.30, 0.47]. Indistinguishable from chance.
- `drought_freq`: −0.13 [−0.37, +0.07], hit rate 0.29 [0.16, 0.44].
- `storage`: −0.10 [−0.21, +0.01], hit rate 0.31 [0.23, 0.40].
- `combined`: −0.11 [−0.34, +0.15], hit rate 0.32 [0.20, 0.45].

**The wrong-way results are not evidence of the opposite ranking either.** The placebo shows the same negative correlations in 2016, 2019 and 2025, years with no drought anywhere in Iowa. A score that "predicts" relative loss in a year without drought is picking up something persistent about counties, not drought vulnerability. Two things are at work: (1) counties with the worst 2000–2015 drought losses or the most frequent drought tended to do relatively better afterwards regardless of weather (regression to the mean); (2) the strict trend extrapolates unevenly. County trend slopes from 2000–2015 correlate +0.70 with storage for corn, so western, high-storage counties get over-predicted yields and systematically negative relative losses in every TEST year. Mean TEST relative loss is +8 pp in the low-storage tertile and −5 pp in the high-storage tertile for corn, drought or not.

**The diagnostic cell confirms both explanations.** With the all-years anomaly as the outcome (used only in that cell), the TEST-year storage slopes sit near zero (corn between −3.2 and +2.0 per 10 mm) while the strict ones run from −0.8 to −9.9, so the strict storage result is an extrapolation artifact. The score correlations in the zero-drought years 2016, 2019 and 2025 stay negative under both outcomes (combined score, corn: −0.34 vs −0.40, −0.46 vs −0.29, −0.29 vs −0.21), so regression to the mean is present as well. Notebook 05 replaces the trend with a local drought penalty to address this.

**Secondary check (spec C storage slope).** TRAIN years: corn −0.01 [−2.04, +1.97], soybeans −0.25 [−2.19, +1.56], consistent with notebook 03's "no clear effect". TEST years: corn −5.49 [−8.20, −2.60], soybeans −1.84 [−4.26, +1.14]. The corn TEST estimate does **not** agree with TRAIN, but it should not be read as a storage effect. The placebo gives the same negative storage slope in zero-drought years (corn 2016 −5.6, 2019 −4.6, 2025 −9.9 per 10 mm), so it is the trend-extrapolation bias described above. A fair out-of-sample storage test needs a detrending method whose extrapolation error is not correlated with storage, which a straight 16-year line is not.

**What we can say to the Soil Conservation Office.**
- We can show *where* and *how often* severe drought occurred in Iowa over 26 years, and which counties lost most in those years. Those are observations, not predictions.
- We cannot say that a county's past drought record, past drought losses, or its soil water storage predicts which counties will lose most in the next drought. On the ten years we held out, none of these rankings did better than picking counties at random.
- We cannot say that higher soil water storage reduces drought-year yield loss in Iowa. Within-sample the association was indistinguishable from zero; out-of-sample the test is confounded by trend extrapolation.

**What we should not claim.** No priority map built from these scores should be presented as "counties most at risk". Any map should be labelled as historical exposure. Claims about storage reducing losses, or about a soil program's effect, are not supported by this data.
